# **Лекция 15. Парный трейдинг по ценам + граф связей рынка**

Логика ноутбука:
1. Загружаем цены 10-30 акций из T-Invest.
2. Строим heatmap и граф связей **по ценам**.
3. Визуализируем граф в 2D и 3D (3D можно вращать мышкой).
4. После анализа связей выбираем пару (автоматически или вручную).
5. Считаем спред как разницу цен, Z-score, сигналы и экспорт JSON.


## **1. Установка библиотек**
Эта ячейка для Colab. Выполняется один раз.


In [ ]:
%pip install -q pandas numpy matplotlib seaborn statsmodels scikit-learn networkx plotly
%pip install -q cachetools==5.5.2 deprecation "protobuf<5" "grpcio>=1.59.3" python-dateutil
%pip install -q --no-deps git+https://github.com/RussianInvestments/invest-python.git@0.2.0-beta97

print("Установка завершена.")


*Промежуточный вывод:* рабочая среда подготовлена.


## **2. Импорт библиотек**


In [ ]:
import os
import json
from datetime import timedelta
from pathlib import Path
from getpass import getpass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import plotly.graph_objects as go

from matplotlib import cm
from matplotlib.colors import TwoSlopeNorm, to_hex

from tinkoff.invest import Client, CandleInterval
from tinkoff.invest.schemas import InstrumentIdType
from tinkoff.invest.utils import now


*Промежуточный вывод:* модули загружены.


## **3. Параметры исследования**


In [ ]:
CLASS_CODE = "TQBR"

TICKERS = [
    "SBER", "AFLT", "GAZP", "LKOH", "ROSN", "NVTK", "MOEX", "MGNT", "TATN", "PLZL",
    "ALRS", "CHMF", "PHOR", "SNGS", "VTBR", "RUAL", "MAGN", "BSPB", "FLOT", "YDEX"
]

DAYS_BACK = 600
CORR_WINDOW = 60
GRAPH_EDGE_THRESHOLD = 0.60
Z_WINDOW = 30
ENTRY_THRESHOLD = 1.0

OUTPUT_DIR = Path("reports/zscore_pair_sber_aflt")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Тикеров:", len(TICKERS))
print("Окно корреляций:", CORR_WINDOW)
print("Порог ребер графа:", GRAPH_EDGE_THRESHOLD)


*Промежуточный вывод:* базовые параметры установлены.


## **4. Токен T-Invest**


In [ ]:
TOKEN = os.environ.get("TINVEST_TOKEN", "").strip()
if not TOKEN:
    TOKEN = getpass("Введите T-Invest токен: ").strip()
    os.environ["TINVEST_TOKEN"] = TOKEN

APP_NAME = "lecture15-price-graph-colab"
print("Token loaded:", bool(TOKEN))


*Промежуточный вывод:* токен подключен, можно загружать цены.


## **5. Функции загрузки цен**


In [ ]:
def quotation_to_float(q):
    return float(q.units + q.nano / 1e9)


def load_close_series(api: Client, ticker: str, class_code: str, days_back: int) -> pd.Series:
    instrument = api.instruments.share_by(
        id_type=InstrumentIdType.INSTRUMENT_ID_TYPE_TICKER,
        class_code=class_code,
        id=ticker,
    ).instrument

    rows = []
    for candle in api.get_all_candles(
        figi=instrument.figi,
        from_=now() - timedelta(days=days_back),
        interval=CandleInterval.CANDLE_INTERVAL_DAY,
    ):
        rows.append({"Date": candle.time, "Close": quotation_to_float(candle.close)})

    df = pd.DataFrame(rows).sort_values("Date").drop_duplicates("Date")
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.set_index("Date")
    if getattr(df.index, "tz", None) is not None:
        df.index = df.index.tz_convert(None)
    return df["Close"].rename(ticker)


*Промежуточный вывод:* функции готовы.


## **6. Загрузка цен по корзине**


In [ ]:
series_list = []
with Client(TOKEN, app_name=APP_NAME) as api:
    for ticker in TICKERS:
        s = load_close_series(api, ticker=ticker, class_code=CLASS_CODE, days_back=DAYS_BACK)
        series_list.append(s)
        print(f"{ticker}: {len(s)} rows")

prices = pd.concat(series_list, axis=1).sort_index()
prices = prices.ffill().dropna()

print("Prices shape:", prices.shape)
display(prices.tail())


*Промежуточный вывод:* получена единая таблица цен.


## **7. Heatmap корреляций по ценам**


In [ ]:
price_corr = prices.tail(CORR_WINDOW).corr()

plt.figure(figsize=(14, 10))
sns.heatmap(price_corr, cmap="RdBu_r", center=0, square=True)
plt.title("Heatmap корреляций по ценам")
plt.tight_layout()
plt.show()


*Промежуточный вывод:* связи считаются по ценам, без доходностей.


## **8. Топ пар по модулю корреляции цен**


In [ ]:
rows = []
cols = list(price_corr.columns)
for i, t1 in enumerate(cols):
    for t2 in cols[i + 1:]:
        corr_val = float(price_corr.loc[t1, t2])
        rows.append({"ticker_1": t1, "ticker_2": t2, "corr": corr_val, "abs_corr": abs(corr_val)})

top_pairs = pd.DataFrame(rows).sort_values("abs_corr", ascending=False).reset_index(drop=True)
display(top_pairs.head(30))


*Промежуточный вывод:* есть таблица кандидатов для выбора пары.


## **9. Почему корреляция может быть отрицательной? (проверка по окнам)**


In [ ]:
check_a = "SBER"
check_b = "AFLT"

for w in [30, 60, 120, 250]:
    c = prices[[check_a, check_b]].tail(w).corr().iloc[0, 1]
    print(f"window={w:>3} -> corr({check_a},{check_b}) = {c:.6f}")


*Промежуточный вывод:* знак корреляции зависит от выбранного окна и текущего режима рынка.


## **10. Граф связей по ценам (NetworkX)**


In [ ]:
G = nx.Graph()
for ticker in price_corr.columns:
    G.add_node(ticker)

for i, t1 in enumerate(price_corr.columns):
    for t2 in price_corr.columns[i + 1:]:
        w = float(price_corr.loc[t1, t2])
        if abs(w) >= GRAPH_EDGE_THRESHOLD:
            G.add_edge(t1, t2, weight=w)

degree_centrality = pd.Series(nx.degree_centrality(G)).sort_values(ascending=False)
print("Nodes:", G.number_of_nodes(), "| Edges:", G.number_of_edges())
display(degree_centrality.head(10).to_frame("degree_centrality"))


*Промежуточный вывод:* граф построен, ключевые узлы определены.


## **11. Улучшенная 2D визуализация**


In [ ]:
plt.figure(figsize=(13, 9))
pos_2d = nx.kamada_kawai_layout(G)

edge_weights = [G[u][v]["weight"] for u, v in G.edges()]
edge_widths = [1 + 5 * abs(w) for w in edge_weights]
edge_colors = ["#1976D2" if w >= 0 else "#D32F2F" for w in edge_weights]

node_degree = dict(G.degree())
node_sizes = [350 + 120 * node_degree[n] for n in G.nodes()]

nx.draw_networkx_nodes(
    G, pos_2d,
    node_size=node_sizes,
    node_color="#FFA726",
    alpha=0.95,
    linewidths=1.0,
    edgecolors="#5D4037",
)
nx.draw_networkx_edges(G, pos_2d, width=edge_widths, edge_color=edge_colors, alpha=0.75)
nx.draw_networkx_labels(G, pos_2d, font_size=9)

plt.title("Граф связей цен (2D)")
plt.axis("off")
plt.show()


*Промежуточный вывод:* 2D граф читается лучше: цвет и толщина ребер отражают связь.


## **12. Интерактивный 3D граф (вращение мышкой)**


In [ ]:
pos_3d = nx.spring_layout(G, seed=42, dim=3)
norm = TwoSlopeNorm(vmin=-1.0, vcenter=0.0, vmax=1.0)
cmap = cm.get_cmap("RdBu_r")

def corr_to_hex(c):
    return to_hex(cmap(norm(float(c))))

edge_traces = []
for u, v, data in G.edges(data=True):
    x0, y0, z0 = pos_3d[u]
    x1, y1, z1 = pos_3d[v]
    w = float(data["weight"])
    edge_traces.append(
        go.Scatter3d(
            x=[x0, x1, None],
            y=[y0, y1, None],
            z=[z0, z1, None],
            mode="lines",
            line={"width": 2 + 6 * abs(w), "color": corr_to_hex(w)},
            hovertext=[f"{u}-{v}: corr={w:.4f}", f"{u}-{v}: corr={w:.4f}", None],
            hoverinfo="text",
            showlegend=False,
        )
    )

node_x, node_y, node_z, node_text = [], [], [], []
for n in G.nodes():
    x, y, z = pos_3d[n]
    node_x.append(x)
    node_y.append(y)
    node_z.append(z)
    node_text.append(f"{n}<br>degree={G.degree(n)}")

node_trace = go.Scatter3d(
    x=node_x, y=node_y, z=node_z,
    mode="markers+text",
    text=list(G.nodes()),
    textposition="top center",
    hovertext=node_text,
    hoverinfo="text",
    marker={
        "size": 9,
        "color": "#FFA726",
        "line": {"color": "#3E2723", "width": 1},
        "opacity": 0.95,
    },
    showlegend=False,
)

fig3d = go.Figure(data=edge_traces + [node_trace])
fig3d.update_layout(
    title="3D граф связей цен (цвет ветвей = корреляция цен)",
    template="plotly_white",
    width=1100,
    height=800,
    margin={"l": 0, "r": 0, "t": 60, "b": 0},
    scene={"xaxis": {"visible": False}, "yaxis": {"visible": False}, "zaxis": {"visible": False}},
)
fig3d.show()

html_path = OUTPUT_DIR / "price_graph_3d.html"
fig3d.write_html(html_path, include_plotlyjs="cdn")
print("3D html saved:", html_path)


*Промежуточный вывод:* в 3D графе цвет ветвей напрямую кодирует корреляцию цен между узлами.


## **13. Выбор пары после анализа графа**


In [ ]:
# Автовыбор самой сильной пары:
PAIR_TICKER_1 = str(top_pairs.iloc[0]["ticker_1"])
PAIR_TICKER_2 = str(top_pairs.iloc[0]["ticker_2"])

# Если хотите, задайте вручную:
# PAIR_TICKER_1, PAIR_TICKER_2 = "SBER", "AFLT"

print("Выбранная пара:", PAIR_TICKER_1, "/", PAIR_TICKER_2)


*Промежуточный вывод:* пара выбирается после анализа структуры связей.


## **14. Подготовка пары и расчет спреда (разница цен)**


In [ ]:
pair_df = prices[[PAIR_TICKER_1, PAIR_TICKER_2]].copy()
pair_df["Spread"] = pair_df[PAIR_TICKER_1] - pair_df[PAIR_TICKER_2]
pair_df["Spread_Mean"] = pair_df["Spread"].rolling(window=Z_WINDOW).mean()
pair_df["Spread_STD"] = pair_df["Spread"].rolling(window=Z_WINDOW).std()
pair_df["Z_Score"] = (pair_df["Spread"] - pair_df["Spread_Mean"]) / pair_df["Spread_STD"]
display(pair_df.tail())


*Промежуточный вывод:* спред и Z-score готовы для сигналов.


## **15. Графики пары (цены, спред, Z-score)**


In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(pair_df.index, pair_df[PAIR_TICKER_1], label=PAIR_TICKER_1)
plt.plot(pair_df.index, pair_df[PAIR_TICKER_2], label=PAIR_TICKER_2)
plt.title(f"Цены пары: {PAIR_TICKER_1} и {PAIR_TICKER_2}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(14, 6))
plt.plot(pair_df.index, pair_df["Spread"], label="Spread", color="tab:blue")
plt.title(f"Spread = {PAIR_TICKER_1} - {PAIR_TICKER_2}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(14, 6))
plt.plot(pair_df.index, pair_df["Z_Score"], label="Z-score", color="tab:purple")
plt.axhline(ENTRY_THRESHOLD, color="red", linestyle="--", label="+entry")
plt.axhline(-ENTRY_THRESHOLD, color="green", linestyle="--", label="-entry")
plt.axhline(0, color="black", linestyle="--")
plt.legend()
plt.grid(True, alpha=0.3)
plt.title("Z-score спреда")
plt.show()


*Промежуточный вывод:* визуально проверили поведение пары и уровни входа.


## **16. Сигналы и простой backtest**


In [ ]:
pair_df["Long_Signal"] = np.where(pair_df["Z_Score"] <= -ENTRY_THRESHOLD, 1, 0)
pair_df["Short_Signal"] = np.where(pair_df["Z_Score"] >= ENTRY_THRESHOLD, -1, 0)
pair_df["Position"] = pair_df["Long_Signal"] + pair_df["Short_Signal"]
pair_df["Position_Lagged"] = pair_df["Position"].shift(1).fillna(0)

pair_df["Spread_Change"] = pair_df["Spread"].diff()
pair_df["Strategy_PnL"] = pair_df["Position_Lagged"] * pair_df["Spread_Change"]
pair_df["Cumulative_PnL"] = pair_df["Strategy_PnL"].fillna(0).cumsum()

plt.figure(figsize=(14, 6))
plt.plot(pair_df.index, pair_df["Cumulative_PnL"], label="Cumulative PnL", color="tab:orange")
plt.title("Backtest по спреду")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


*Промежуточный вывод:* стратегия рассчитана и протестирована на истории.


## **17. Финальный сигнал**


In [ ]:
last_row = pair_df.dropna(subset=["Z_Score"]).iloc[-1]
last_date = pd.Timestamp(last_row.name).normalize()
last_z = float(last_row["Z_Score"])

if last_z <= -ENTRY_THRESHOLD:
    raw_signal = "LONG_SPREAD"
    action = "BUY_SPREAD"
elif last_z >= ENTRY_THRESHOLD:
    raw_signal = "SHORT_SPREAD"
    action = "SELL_SPREAD"
else:
    raw_signal = "HOLD"
    action = "HOLD"

pair_corr_value = float(price_corr.loc[PAIR_TICKER_1, PAIR_TICKER_2])
centrality_1 = float(degree_centrality.get(PAIR_TICKER_1, 0.0))
centrality_2 = float(degree_centrality.get(PAIR_TICKER_2, 0.0))

print("Signal date:", last_date.date())
print("Last Z-score:", round(last_z, 6))
print("Signal:", raw_signal)
print("Action:", action)
print("Pair price corr:", round(pair_corr_value, 6))


*Промежуточный вывод:* итоговый сигнал сформирован.


## **18. Экспорт JSON для бота**


In [ ]:
signal_payload = {
    "strategy": "pair_zscore_price_graph",
    "signal_date": str(last_date.date()),
    "class_code": CLASS_CODE,
    "pair": [PAIR_TICKER_1, PAIR_TICKER_2],
    "leg1_ticker": PAIR_TICKER_1,
    "leg2_ticker": PAIR_TICKER_2,
    "price_corr_window": int(CORR_WINDOW),
    "price_corr_value": pair_corr_value,
    "spread_value": float(last_row["Spread"]),
    "current_z_score": last_z,
    "entry_threshold": float(ENTRY_THRESHOLD),
    "raw_signal": raw_signal,
    "action": action,
    "leg1_degree_centrality": centrality_1,
    "leg2_degree_centrality": centrality_2,
}

signal_path = OUTPUT_DIR / "latest_forecast_signal_pair_zscore.json"
signal_path.write_text(json.dumps(signal_payload, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved:", signal_path)
print(signal_path.read_text(encoding="utf-8"))


*Промежуточный вывод:* JSON сигнал готов к запуску через `run_trade_signal.py`.
